# ELE6310 - Assignment 2 - Data-Flow and Design Space

**bold text**#### Name: Nicolas Billy
#### Student ID: 2508164

In [3]:
#@title Mount your Google Drive
%matplotlib inline
from importlib import reload

from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

Mounted at /content/gdrive


In [4]:
#@title Link your assignment folder & install requirements
#@markdown Enter the path to the assignment folder in your Google Drive
import sys
import os
import shutil
import warnings

folder = "/content/gdrive/MyDrive/A2" #@param {type:"string"}

# Ensure clean symbolic link creation and list contents for verification
!ln -Ts $folder /content/A2 2> /dev/null

# Add the assignment folder to Python path
if '/content/A2' not in sys.path:
    sys.path.insert(0, '/content/A2') # Corrected typo: sys.path.insert

# Install requirements
!pip install -qr /content/A2/requirements.txt

# Check if CUDA is available
import torch
if not torch.cuda.is_available():
    warnings.warn('CUDA is not available.')

In [6]:
!pip install -qr /content/A2/requirements.txt

## Q1- Memory-access simulation

le code est accessible au lien suivant:

https://github.com/nisaba16/DNN_hardware/tree/main/A2

![Texte alternatif](table2.png)

Comparison with timeloop results

In [69]:
import pandas as pd

data = {
    "Dataflow":        ["OS Tiled (PRM)", "OS Tiled (MRP)", "OS Untiled (PRM)", "OS Untiled (MRP)", "Weight Stationary"],
    "Timeloop (pJ)":   [12_137, 12_489, 16_357, 23_739, 13_946],
    "Simulator (pJ)":  [18_585, 11_081, 18_585, 11_574, 20_859],
}

df = pd.DataFrame(data)
df["Diff (%)"] = ((df["Simulator (pJ)"] - df["Timeloop (pJ)"]) / df["Timeloop (pJ)"] * 100).round(1)
df

,Dataflow,Timeloop (pJ),Simulator (pJ),Diff (%)
0,OS Tiled (PRM),12137,18585,53.1
1,OS Tiled (MRP),12489,11081,-11.3
2,OS Untiled (PRM),16357,18585,13.6
3,OS Untiled (MRP),23739,11574,-51.2
4,Weight Stationary,13946,20859,49.6


## Q2 - Energy Model With Accelergy

In [3]:
import math

SRAM_RD, SRAM_WR = 7.95, 5.45    
REG_RD, REG_WR = 0.42, 0.42       

def sram(depth, width):
    w = width / 8
    d = 1 + 0.1 * math.log2(depth / 32768) if depth != 32768 else 1
    return SRAM_RD * w * max(0.5, d), SRAM_WR * w * max(0.5, d)

def regfile(depth, width):
    w = width / 8
    d = math.sqrt(depth / 64)
    return REG_RD * w * d, REG_WR * w * d

a)

In [4]:
for w in [16, 32, 64, 128, 256, 512, 1024, 2048]:
    r, wr = sram(8192, w)
    print(f"  width={w:4}: read={r:.2f} pJ, write={wr:.2f} pJ")

  width=  16: read=12.72 pJ, write=8.72 pJ
  width=  32: read=25.44 pJ, write=17.44 pJ
  width=  64: read=50.88 pJ, write=34.88 pJ
  width= 128: read=101.76 pJ, write=69.76 pJ
  width= 256: read=203.52 pJ, write=139.52 pJ
  width= 512: read=407.04 pJ, write=279.04 pJ
  width=1024: read=814.08 pJ, write=558.08 pJ
  width=2048: read=1628.16 pJ, write=1116.16 pJ


In [5]:
for d in [512, 1024, 2048, 4096, 8192]:
    r, wr = sram(d, 128)
    print(f"  depth={d:4}: read={r:.2f} pJ, write={wr:.2f} pJ")

  depth= 512: read=63.60 pJ, write=43.60 pJ
  depth=1024: read=63.60 pJ, write=43.60 pJ
  depth=2048: read=76.32 pJ, write=52.32 pJ
  depth=4096: read=89.04 pJ, write=61.04 pJ
  depth=8192: read=101.76 pJ, write=69.76 pJ


b)

In [6]:
for w in [16, 32, 64, 128, 256, 512, 1024, 2048]:
    r, wr = regfile(8192, w)
    print(f"  width={w:4}: read={r:.2f} pJ, write={wr:.2f} pJ")

  width=  16: read=9.50 pJ, write=9.50 pJ
  width=  32: read=19.01 pJ, write=19.01 pJ
  width=  64: read=38.01 pJ, write=38.01 pJ
  width= 128: read=76.03 pJ, write=76.03 pJ
  width= 256: read=152.06 pJ, write=152.06 pJ
  width= 512: read=304.11 pJ, write=304.11 pJ
  width=1024: read=608.22 pJ, write=608.22 pJ
  width=2048: read=1216.45 pJ, write=1216.45 pJ


## Q3- Structured Pruning

* First, complete `model_to_spars` and `generate_resnet_layers` in `solution. py`.

In [7]:
import solution
import torch
import numpy as np
import random
from matplotlib import pyplot as plt
import os

In [8]:
from common.utils import load_CIFAR10_dataset, evaluate, fit, model_size
from common.resnet import resnet32

In [9]:
Seed = 6310
torch.manual_seed(Seed)
np.random.seed(Seed)
random.seed(Seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(Seed)
    torch.cuda.manual_seed_all(Seed)

In [16]:
train_loader, test_loader, calibration_loader = load_CIFAR10_dataset(batch_size=256, calibration_batch_size=1024)
model = resnet32(pretrained=True, save_path='./save/')
device = torch.device('cuda:0')
model.to(device)

accuracy = evaluate(model, test_loader, device)
print("test accuracy of fp model:", accuracy, "model size:", model_size(model))

test accuracy of fp model: 93.52999877929688 model size: model size: 1.781MB


* Use a method of your choice to find the optimal energy consumption for ResNet-32 with a constraint on test accuracy above 85%.

Any reasonable attempt at exploring the design space will give you full marks. Better approaches/results will be considered for bonus points.

In [11]:

prune_ratio_dict = {
    'conv1': 0.3,

    'layer1.0.conv1': 0.3,
    'layer1.0.conv2': 0.3,
    'layer1.1.conv1': 0.3,
    'layer1.1.conv2': 0.3,
    'layer1.2.conv1': 0.3,
    'layer1.2.conv2': 0.3,
    'layer1.3.conv1': 0.3,
    'layer1.3.conv2': 0.3,
    'layer1.4.conv1': 0.3,
    'layer1.4.conv2': 0.3,

    'layer2.0.conv1': 0.1,
    'layer2.0.conv2': 0.1,
    'layer2.0.downsample.0': 0.1,
    'layer2.1.conv1': 0.1,
    'layer2.1.conv2': 0.1,
    'layer2.2.conv1': 0.1,
    'layer2.2.conv2': 0.1,
    'layer2.3.conv1': 0.1,
    'layer2.3.conv2': 0.1,
    'layer2.4.conv1': 0.1,
    'layer2.4.conv2': 0.1,

    'layer3.0.conv1': 0.3,
    'layer3.0.conv2': 0.3,
    'layer3.0.downsample.0': 0.3,
    'layer3.1.conv1': 0.3,
    'layer3.1.conv2': 0.3,
    'layer3.2.conv1': 0.3,
    'layer3.2.conv2': 0.3,
    'layer3.3.conv1': 0.3,
    'layer3.3.conv2': 0.3,
    'layer3.4.conv1': 0.3,
    'layer3.4.conv2': 0.3
}

print("Pruning Strategy: Layer1=30%, Layer2=10%, Layer3=30%")
print("Layer2 pruning ratios:", {k: v for k, v in prune_ratio_dict.items() if 'layer2' in k})

Pruning Strategy: Layer1=30%, Layer2=10%, Layer3=30%
Layer2 pruning ratios: {'layer2.0.conv1': 0.1, 'layer2.0.conv2': 0.1, 'layer2.0.downsample.0': 0.1, 'layer2.1.conv1': 0.1, 'layer2.1.conv2': 0.1, 'layer2.2.conv1': 0.1, 'layer2.2.conv2': 0.1, 'layer2.3.conv1': 0.1, 'layer2.3.conv2': 0.1, 'layer2.4.conv1': 0.1, 'layer2.4.conv2': 0.1}


In [ ]:
if 'model' not in locals():
    model = resnet32(pretrained=True, save_path='./save/')
    model.to(device)
    base_accuracy = evaluate(model, test_loader, device)
    print(f"Base model accuracy: {base_accuracy:.2f}%")

### Pruning Exploration per layer

#### Step 1

We first explore single random pruning of individual elements without fine tuning to check what weights are impacted by this.

#### Step 2

from this explore around and retrain from the top 10.



In [59]:

import random
import gc
from tqdm import tqdm

def create_random_per_layer_dict(values=[0, 0.1, 0.2]):

    layers = [
        'layer1.0.conv1', 'layer1.0.conv2', 'layer1.1.conv1', 'layer1.1.conv2',
        'layer1.2.conv1', 'layer1.2.conv2', 'layer1.3.conv1', 'layer1.3.conv2',
        'layer1.4.conv1', 'layer1.4.conv2',
        'layer2.0.conv1', 'layer2.0.conv2', 'layer2.1.conv1', 'layer2.1.conv2',
        'layer2.2.conv1', 'layer2.2.conv2', 'layer2.3.conv1', 'layer2.3.conv2',
        'layer2.4.conv1', 'layer2.4.conv2',
        'layer3.0.conv1', 'layer3.0.conv2', 'layer3.1.conv1', 'layer3.1.conv2',
        'layer3.2.conv1', 'layer3.2.conv2', 'layer3.3.conv1', 'layer3.3.conv2',
        'layer3.4.conv1', 'layer3.4.conv2'
    ]

    prune_dict = {
        'conv1': 0,
        'layer2.0.downsample.0': 0,
        'layer3.0.downsample.0': 0
    }


    for layer in layers:
        prune_dict[layer] = random.choice(values)

    return prune_dict

def explore_random_configs(model, n_iterations=100):
  results = []

  for i in tqdm(range(n_iterations)):
      torch.cuda.empty_cache()
      gc.collect()

      current_config = create_random_per_layer_dict(values=[0,0.1, 0.2,0.3,0.5])

      pruned_model = solution.model_to_spars(model, current_config)

      acc = evaluate(pruned_model, test_loader, device)
      size = model_size(pruned_model)

      results.append({
          'accuracy': acc,
          'size': size,
          'config': current_config
      })

      del pruned_model

  # on garde les 10 meilleures
  top_20 = sorted(results, key=lambda x: x['accuracy'], reverse=True)[:20]

  for idx, res in enumerate(top_20):
      print(f"Rang {idx+1}: Accuracy {res['accuracy']:.2f}% | Taille {res['size']}")

  return top_20

top_20 = explore_random_configs( model, n_iterations=100)

100%|██████████| 100/100 [04:16<00:00,  2.56s/it]

Rang 1: Accuracy 57.03% | Taille model size: 1.489MB
Rang 2: Accuracy 40.63% | Taille model size: 1.479MB
Rang 3: Accuracy 35.30% | Taille model size: 1.569MB
Rang 4: Accuracy 31.81% | Taille model size: 1.479MB
Rang 5: Accuracy 30.58% | Taille model size: 1.546MB
Rang 6: Accuracy 29.44% | Taille model size: 1.460MB
Rang 7: Accuracy 28.63% | Taille model size: 1.501MB
Rang 8: Accuracy 27.99% | Taille model size: 1.472MB
Rang 9: Accuracy 25.70% | Taille model size: 1.447MB
Rang 10: Accuracy 25.52% | Taille model size: 1.472MB
Rang 11: Accuracy 23.79% | Taille model size: 1.461MB
Rang 12: Accuracy 23.64% | Taille model size: 1.551MB
Rang 13: Accuracy 23.37% | Taille model size: 1.487MB
Rang 14: Accuracy 22.69% | Taille model size: 1.420MB
Rang 15: Accuracy 21.19% | Taille model size: 1.439MB
Rang 16: Accuracy 20.74% | Taille model size: 1.323MB
Rang 17: Accuracy 19.76% | Taille model size: 1.397MB
Rang 18: Accuracy 19.45% | Taille model size: 1.633MB
Rang 19: Accuracy 19.01% | Taille mod

In [66]:
top_20 = sorted(top_20, key=lambda x: x['size'], reverse=False)

In [67]:

for mod in top_20:
  print( mod['size'])

model size: 1.323MB
model size: 1.339MB
model size: 1.360MB
model size: 1.397MB
model size: 1.420MB
model size: 1.439MB
model size: 1.447MB
model size: 1.460MB
model size: 1.461MB
model size: 1.472MB
model size: 1.472MB
model size: 1.479MB
model size: 1.479MB
model size: 1.487MB
model size: 1.489MB
model size: 1.501MB
model size: 1.546MB
model size: 1.551MB
model size: 1.569MB
model size: 1.633MB


Avec entrainement, on repart de ces modèles, des 3 plus petits en terme de taille.

In [68]:

for i in range(3):
    prune_dict = top_20[i]['config']
    torch.cuda.empty_cache()
    gc.collect()

    sparsed_model = solution.model_to_spars(model, prune_dict)
    accuracy = evaluate(sparsed_model, test_loader, device)
    size = model_size(sparsed_model)

    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(sparsed_model.parameters(), 1e-3, momentum=0.9, weight_decay=0.0005, nesterov=True)
    scheduler = None
    train_accuracy, test_accuracy = fit(sparsed_model, 5, train_loader, test_loader, criterion, optimizer, scheduler, device)

    print(size)
    print(test_accuracy)

fit:  20%|██        | 1/5 [00:48<03:14, 48.54s/it]

epoch 1: train_accuracy=96.35757446289062, test_accuracy=90.26000213623047


fit:  40%|████      | 2/5 [01:34<02:20, 46.77s/it]

epoch 2: train_accuracy=97.48445892333984, test_accuracy=90.88999938964844


fit:  60%|██████    | 3/5 [02:19<01:31, 45.99s/it]

...

fit:  60%|██████    | 3/5 [02:13<01:28, 44.43s/it]

epoch 3: train_accuracy=96.76046752929688, test_accuracy=90.6500015258789


fit:  80%|████████  | 4/5 [02:57<00:44, 44.46s/it]

epoch 4: train_accuracy=97.08769989013672, test_accuracy=90.81999969482422


epoch 5: train_accuracy=97.49059295654297, test_accuracy=91.00999450683594
model size: 1.360MB
[89.27999877929688, 90.09000396728516, 90.6500015258789, 90.81999969482422, 91.00999450683594]


**Conclusion**

We take the smallest one:


In [72]:
prune_dict = top_20[0]['config']

print(prune_dict)

{'conv1': 0, 'layer2.0.downsample.0': 0, 'layer3.0.downsample.0': 0, 'layer1.0.conv1': 0, 'layer1.0.conv2': 0.3, 'layer1.1.conv1': 0.2, 'layer1.1.conv2': 0.1, 'layer1.2.conv1': 0.3, 'layer1.2.conv2': 0, 'layer1.3.conv1': 0.3, 'layer1.3.conv2': 0.5, 'layer1.4.conv1': 0.3, 'layer1.4.conv2': 0.3, 'layer2.0.conv1': 0.5, 'layer2.0.conv2': 0.3, 'layer2.1.conv1': 0, 'layer2.1.conv2': 0.3, 'layer2.2.conv1': 0, 'layer2.2.conv2': 0.1, 'layer2.3.conv1': 0, 'layer2.3.conv2': 0.5, 'layer2.4.conv1': 0.5, 'layer2.4.conv2': 0.5, 'layer3.0.conv1': 0.2, 'layer3.0.conv2': 0, 'layer3.1.conv1': 0.5, 'layer3.1.conv2': 0.3, 'layer3.2.conv1': 0.1, 'layer3.2.conv2': 0.5, 'layer3.3.conv1': 0, 'layer3.3.conv2': 0, 'layer3.4.conv1': 0.5, 'layer3.4.conv2': 0.5}


In [75]:
sparsed_model = solution.model_to_spars(model, prune_dict)
accuracy = evaluate(sparsed_model, test_loader, device)

size = model_size(sparsed_model)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(sparsed_model.parameters(), 1e-3, momentum=0.9, weight_decay=0.0005, nesterov=True)
scheduler = None
train_accuracy, test_accuracy = fit(sparsed_model, 5, train_loader, test_loader, criterion, optimizer, scheduler, device)


fit:  20%|██        | 1/5 [00:44<02:59, 44.84s/it]

epoch 1: train_accuracy=96.47209930419922, test_accuracy=90.3499984741211


fit:  40%|████      | 2/5 [01:29<02:14, 44.84s/it]

epoch 2: train_accuracy=97.33106994628906, test_accuracy=90.87999725341797


fit:  60%|██████    | 3/5 [02:14<01:29, 45.00s/it]

epoch 3: train_accuracy=97.95280456542969, test_accuracy=91.2699966430664


fit:  80%|████████  | 4/5 [03:00<00:45, 45.18s/it]

epoch 4: train_accuracy=98.2145767211914, test_accuracy=91.45999908447266


epoch 5: train_accuracy=98.26570892333984, test_accuracy=91.33999633789062


In [81]:
model_size(model)

'model size: 1.781MB'

In [80]:
model_size(sparsed_model)

'model size: 1.323MB'

In [83]:
reduction = (1.781-1.323)/1.781 *100
print(reduction)

25.715889949466593


After fine-tuning, save the model and generate the YAML files for each layers of the pruned network. Then you can use `run_Accelergy` to estimate the energy consumption of pruned network.

In [88]:
solution.generate_resnet_layers(sparsed_model, base_path='A2/common/layer_prob_base.yaml',  path='/content/A2/Q3/prob')

In [89]:
# List the contents of the generated directory
!ls -F /content/A2/Q3/prob/

conv1.yaml	     layer2_0_conv1.yaml	 layer3_0_conv1.yaml
layer1_0_conv1.yaml  layer2_0_conv2.yaml	 layer3_0_conv2.yaml
layer1_0_conv2.yaml  layer2_0_downsample_0.yaml  layer3_0_downsample_0.yaml
layer1_1_conv1.yaml  layer2_1_conv1.yaml	 layer3_1_conv1.yaml
layer1_1_conv2.yaml  layer2_1_conv2.yaml	 layer3_1_conv2.yaml
layer1_2_conv1.yaml  layer2_2_conv1.yaml	 layer3_2_conv1.yaml
layer1_2_conv2.yaml  layer2_2_conv2.yaml	 layer3_2_conv2.yaml
layer1_3_conv1.yaml  layer2_3_conv1.yaml	 layer3_3_conv1.yaml
layer1_3_conv2.yaml  layer2_3_conv2.yaml	 layer3_3_conv2.yaml
layer1_4_conv1.yaml  layer2_4_conv1.yaml	 layer3_4_conv1.yaml
layer1_4_conv2.yaml  layer2_4_conv2.yaml	 layer3_4_conv2.yaml


You can download individual files by clicking the folder icon on the left sidebar, navigating to `/content/A2/Q3/prob/`, right-clicking the file, and selecting 'Download'.

To download the entire `prob` directory as a zip file, you can use the following command: